# 1.자세감지
- 몇 가지 주요 자세 랜드마크를 사용하여 신체의 존재를 감지
- 포즈 랜드마크 확인 : https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker?hl=ko#pose_landmarker_model

In [1]:
# OpenCV: 웹캠 입력/화면 출력
import cv2

# MediaPipe: 자세 랜드마크 추론/이미지 래핑
import mediapipe as mp

# PoseLandmarker 준비
from pathlib import Path
import urllib.request

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [2]:
# 모델 저장 폴더
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# 웹캠 실습용 lite 모델
POSE_LANDMARKER_MODEL = MODEL_DIR / "pose_landmarker_lite.task"
POSE_LANDMARKER_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"

In [3]:
def download_model(model_path, model_url):
    # 기존 모델은 다운로드 생략
    if model_path.exists():
        return

    # 모델이 없을 때만 다운로드
    print(f"모델 다운로드 중: {model_path}")
    urllib.request.urlretrieve(model_url, model_path)


# 첫 실행 시 자세 모델 다운로드
download_model(POSE_LANDMARKER_MODEL, POSE_LANDMARKER_MODEL_URL)

모델 다운로드 중: models\pose_landmarker_lite.task


In [4]:
# Tasks API 클래스 별칭
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions
mp_pose = vision.PoseLandmark
pose_connections = vision.PoseLandmarksConnections
mp_drawing = vision.drawing_utils

# 랜드마크 점/선 스타일
drawing_spec1 = mp_drawing.DrawingSpec(thickness=1, circle_radius=1, color=(0, 255, 0))
drawing_spec2 = mp_drawing.DrawingSpec(thickness=4, circle_radius=1, color=(0, 0, 255))

In [5]:
def create_pose_landmarker(num_poses=1):
    # 자세 랜드마크 검출기 생성
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(POSE_LANDMARKER_MODEL)),
        running_mode=VisionRunningMode.VIDEO,
        num_poses=num_poses,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5
    )
    return PoseLandmarker.create_from_options(options)


def draw_pose_landmarks(image, pose_result):
    # 사람마다 33개 자세 랜드마크 표시
    for pose_landmarks in pose_result.pose_landmarks:
        # pose_landmark는 어깨, 팔꿈치, 무릎 등 관절 연결 정보를 담고 있음
        mp_drawing.draw_landmarks(
            image,
            pose_landmarks,
            pose_connections.POSE_LANDMARKS,
            drawing_spec1,
            drawing_spec2
        )


In [6]:
# 0번 카메라 사용
cap = cv2.VideoCapture(0)

# FPS 없으면 30 사용
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_pose_landmarker(num_poses=1) as pose_landmarker:
    while cap.isOpened():
        # 웹캠 프레임 읽기
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        # 거울 보기용 좌우 반전
        image = cv2.flip(image, 1)

        # BGR -> RGB 변환
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        # 프레임 timestamp 계산
        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 현재 프레임 자세 랜드마크 검출
        result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

        # 신체 관절/연결선 표시
        draw_pose_landmarks(image, result)

        cv2.imshow("frame", image)
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# 2. 앉은 자세 & 선 자세

In [7]:
def detect_pose(landmarks):
    # 33개 자세 랜드마크 입력
    # 엉덩이/무릎 y 좌표로 서기/앉기 판정

    # 엉덩이/무릎 랜드마크 선택
    left_hip = landmarks[mp_pose.LEFT_HIP.value]
    right_hip = landmarks[mp_pose.RIGHT_HIP.value]
    left_knee = landmarks[mp_pose.LEFT_KNEE.value]
    right_knee = landmarks[mp_pose.RIGHT_KNEE.value]

    # 엉덩이/무릎 평균 y 좌표
    hip_y = (left_hip.y + right_hip.y) / 2
    knee_y = (left_knee.y + right_knee.y) / 2

    # Standing/Sitting 판정
    # 엉덩이가 무릎보다 충분히 위에 있으면 서 있는 자세로 판단
    # 0.9는 약간의 오차를 허용하기 위해서
    if hip_y < knee_y :
        return "Standing"
    else :
        return "Sitting"

In [8]:
# 랜드마크와 Standing/Sitting 결과 표시 예제
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_pose_landmarker(num_poses=1) as pose_landmarker:
    while cap.isOpened():
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        image = cv2.flip(image, 1)
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        # 자세 랜드마크 검출
        result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks:
            # 랜드마크와 연결선을 먼저 화면에 그림
            draw_pose_landmarks(image, result)

            # num_pose=1 이므로 첫 번째 사람의 랜드마크만 자세 판정에 사용
            pose_state = result.pose_landmarks[0]

            # Standing or Sitting 판단
            detect_result = detect_pose(pose_state)

            cv2.putText(image, detect_result, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2, cv2.LINE_AA)

        cv2.imshow("frame", image)
        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

# 팔굽혀펴기 Good, Try pose

In [11]:
import math

def calculate_angle(a, b, c):
    # b를 중심으로 a-b-c 각도 계산
    ba_x = a.x - b.x
    ba_y = a.y - b.y
    bc_x = c.x - b.x
    bc_y = c.y - b.y

    dot = ba_x * bc_x + ba_y * bc_y
    mag_ba = math.sqrt(ba_x**2 + ba_y**2)
    mag_bc = math.sqrt(bc_x**2 + bc_y**2)

    if mag_ba == 0 or mag_bc == 0:
        return 0

    cos_angle = dot / (mag_ba * mag_bc)
    cos_angle = max(-1, min(1, cos_angle))

    return math.degrees(math.acos(cos_angle))


def detect_pushup_pose(landmarks):
    left_shoulder = landmarks[mp_pose.LEFT_SHOULDER.value]
    right_shoulder = landmarks[mp_pose.RIGHT_SHOULDER.value]

    left_elbow = landmarks[mp_pose.LEFT_ELBOW.value]
    right_elbow = landmarks[mp_pose.RIGHT_ELBOW.value]

    left_wrist = landmarks[mp_pose.LEFT_WRIST.value]
    right_wrist = landmarks[mp_pose.RIGHT_WRIST.value]

    left_hip = landmarks[mp_pose.LEFT_HIP.value]
    right_hip = landmarks[mp_pose.RIGHT_HIP.value]

    left_ankle = landmarks[mp_pose.LEFT_ANKLE.value]
    right_ankle = landmarks[mp_pose.RIGHT_ANKLE.value]

    left_elbow_angle = calculate_angle(left_shoulder, left_elbow, left_wrist)
    right_elbow_angle = calculate_angle(right_shoulder, right_elbow, right_wrist)
    elbow_angle = (left_elbow_angle + right_elbow_angle) / 2

    shoulder_y = (left_shoulder.y + right_shoulder.y) / 2
    ankle_y = (left_ankle.y + right_ankle.y) / 2

    # 몸이 바닥과 비슷하게 누워 있는지
    body_horizontal = abs(shoulder_y - ankle_y) < 0.35

    # 팔꿈치 각도 기준
    arms_straight = elbow_angle > 155   # 팔을 거의 다 편 상태
    arms_bent = elbow_angle < 100       # 바닥에 거의 내려간 상태

    if body_horizontal and (arms_straight or arms_bent):
        return "Good pose", elbow_angle
    else:
        return "Try pose", elbow_angle


# 팔굽혀펴기 자세 판정 예제
cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 30
frame_index = 0

with create_pose_landmarker(num_poses=1) as pose_landmarker:
    while cap.isOpened():
        res, image = cap.read()
        if not res:
            print('웹캠에서 이미지를 읽지 못했습니다.')
            break

        image = cv2.flip(image, 1)
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        timestamp_ms = int(frame_index * 1000 / fps)
        frame_index += 1

        result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks:
            draw_pose_landmarks(image, result)

            pose_state = result.pose_landmarks[0]
            detect_result, elbow_angle = detect_pushup_pose(pose_state)

            if detect_result == "Good pose":
                color = (0, 255, 0)
            else:
                color = (0, 0, 255)

            cv2.putText(
                image,
                detect_result,
                (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                color,
                2,
                cv2.LINE_AA
            )

            cv2.putText(
                image,
                f"Elbow angle: {elbow_angle:.1f}",
                (50, 90),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )
        else:
            cv2.putText(
                image,
                "No pose detected",
                (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 255),
                2,
                cv2.LINE_AA
            )

        cv2.imshow("Push-up Pose Detector", image)

        if cv2.waitKey(1) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()